In [1]:
pip install pandas numpy scikit-learn tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers

C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framewo

In [2]:
df = pd.read_csv("hbl_merged_dataset_clean.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df.head()

,date,Open,High,Low,Close,Volume,daily_sentiment,daily_sentiment_sum,daily_news_count,daily_pos,daily_neg,daily_neu,quarterly_sentiment,annual_sentiment,log_return,close_next,log_return_next,direction_next,quarterly_date,annual_date
0,2015-12-31,105.464476,105.985973,104.821817,105.417061,779200,0.0,0.0,0,0,0,0,0.174584,-0.811481,-0.004587,105.917480,0.004736,1,2015-09-30,2015-12-31
1,2016-01-01,105.617218,106.349427,105.617218,105.917480,50100,0.0,0.0,0,0,0,0,0.174584,-0.811481,0.004736,105.390739,-0.004986,0,2015-09-30,2015-12-31
2,2016-01-04,105.933311,106.144016,105.095742,105.390739,209800,0.0,0.0,0,0,0,0,0.174584,-0.811481,-0.004986,105.359093,-0.000300,0,2015-09-30,2015-12-31
3,2016-01-05,104.827059,106.064969,103.778785,105.359093,85500,0.0,0.0,0,0,0,0,0.174584,-0.811481,-0.000300,104.526840,-0.007931,0,2015-09-30,2015-12-31
4,2016-01-06,105.880641,105.880641,104.300333,104.526840,531300,0.0,0.0,0,0,0,0,0.174584,-0.811481,-0.007931,103.373199,-0.011098,0,2015-09-30,2015-12-31


In [3]:
# Returns
df["ret_1"] = df["log_return"]
df["ret_5"] = np.log(df["Close"]).diff(5)
df["ret_10"] = np.log(df["Close"]).diff(10)

# Volatility
df["vol_5"] = df["log_return"].rolling(5).std()
df["vol_10"] = df["log_return"].rolling(10).std()
df["vol_20"] = df["log_return"].rolling(20).std()

# Price structure
df["hl_pct"] = (df["High"] - df["Low"]) / df["Close"]
df["oc_pct"] = (df["Close"] - df["Open"]) / df["Open"]

# Volume
df["log_volume"] = np.log1p(df["Volume"])

# Sentiment rollups
df["sent_3"] = df["daily_sentiment"].rolling(3).mean()
df["sent_7"] = df["daily_sentiment"].rolling(7).mean()

df["news_shock"] = df["daily_sentiment"] * np.log1p(df["daily_news_count"])

# Targets: k-day ahead log return (less noisy than 1-day)
df["target_5d"]  = np.log(df["Close"]).shift(-5)  - np.log(df["Close"])
df["target_10d"] = np.log(df["Close"]).shift(-10) - np.log(df["Close"])

# Drop rows with NaNs (from rolling features + future targets)
df = df.dropna().reset_index(drop=True)


In [4]:
FEATURES = [
    "ret_1","ret_5","ret_10",
    "vol_5","vol_10","vol_20",
    "hl_pct","oc_pct",
    "log_volume",
    "daily_sentiment","daily_news_count","news_shock",
    "sent_3","sent_7",
    "quarterly_sentiment","annual_sentiment"
]

# Choose prediction horizon to reduce noise
HORIZON_DAYS = 5  # set to 10 for 10-day ahead

TARGET = f"target_{HORIZON_DAYS}d"  # "target_5d" or "target_10d"

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)


In [5]:
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val     = X[train_end:val_end], y[train_end:val_end]
X_test, y_test   = X[val_end:], y[val_end:]


In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [7]:
def make_sequences(X, y, seq_len=30):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

SEQ_LEN = 30

Xtr, ytr = make_sequences(X_train, y_train, SEQ_LEN)
Xva, yva = make_sequences(X_val, y_val, SEQ_LEN)
Xte, yte = make_sequences(X_test, y_test, SEQ_LEN)

Xtr.shape

(1701, 30, 16)

In [8]:
model = tf.keras.Sequential([
    layers.Input(shape=(SEQ_LEN, Xtr.shape[-1])),

    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Dropout(0.2),

    layers.Bidirectional(layers.LSTM(32)),
    layers.Dropout(0.2),

    layers.Dense(32, activation="relu"),
    layers.Dense(1)  # regression output
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)        │ (None, 30, 128)             │          41,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 30, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 64)                  │          41,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 84,801 (331.25 KB)

 Trainable params: 84,801 (331.25 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=5,
        factor=0.5
    ),
    # Save the best model for this horizon without overwriting your old 1-day model
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f"bilstm_return_regression_{HORIZON_DAYS}d_best.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    Xtr, ytr,
    validation_data=(Xva, yva),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - loss: 0.0065 - mae: 0.0605 - val_loss: 0.0057 - val_mae: 0.0554 - learning_rate: 0.0010
Epoch 2/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0028 - mae: 0.0409 - val_loss: 0.0034 - val_mae: 0.0421 - learning_rate: 0.0010
Epoch 3/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0021 - mae: 0.0345 - val_loss: 0.0034 - val_mae: 0.0402 - learning_rate: 0.0010
Epoch 4/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0019 - mae: 0.0332 - val_loss: 0.0037 - val_mae: 0.0437 - learning_rate: 0.0010
Epoch 5/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0016 - mae: 0.0308 - val_loss: 0.0032 - val_mae: 0.0390 - learning_rate: 0.0010
Epoch 6/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0015 - mae: 0.0299 - val_loss: 0.0035 - val_mae: 0.0426 - learning_rate: 0.0010
Epoch 7/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0014 - mae: 0.0294 - val_loss: 0.0036 - val_mae: 0.0427 - learning_rate: 0.0010
Epoch 

In [10]:
pred = model.predict(Xte).flatten()

rmse = np.sqrt(mean_squared_error(yte, pred))
mae  = mean_absolute_error(yte, pred)
r2   = r2_score(yte, pred)

rmse, mae, r2

11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step


(np.float64(0.06596419514933392), 0.05184301733970642, -0.20550262928009033)

In [11]:
np.corrcoef(pred, yte)[0,1]

np.float64(0.08482217907828028)

In [12]:
# Final save (weights in memory after training / early stopping)
model.save(f"bilstm_return_regression_{HORIZON_DAYS}d_final.keras")

import joblib
joblib.dump(scaler, f"feature_scaler_{HORIZON_DAYS}d.pkl")


['feature_scaler_5d.pkl']